# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Manual IR and Inverted Matrix coding + Markdown documentation (work as teams)
3. **Push to GitHub** *(15 min)* – Teams commit and push initial notebooks. **Make sure to include your names so it is easy to identify the team that developed the Min-Max code**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(15 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. **Push to the course GitHub** *(automated)* – Find the last code cell in this notebook, update your team number, and run the cell. It will push your notebook to the repo at `submissions/team#`

## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and Inverted Index.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


### Our Document Collection

We chose blog posts from the Python Software Foundation as our dataset. Each blog post counts as one document. We combine the title and the body so that both can be searched later.

We collect the posts from an RSS feed. An RSS feed is a source that lists published posts and provides information such as their titles and content. The posts can include HTML tags, so we remove the tags and keep the readable text.

We store the cleaned posts in a Python list called `documents`. Before moving to tokenization, we check that the list has at least 20 documents and that the collection contains more than 2,000 unique words.

In [1]:
import re

import feedparser
import requests
from bs4 import BeautifulSoup

rss_url = "https://pyfound.blogspot.com/feeds/posts/default?alt=rss"

# Download and read the RSS feed.
response = requests.get(rss_url, timeout=30)
response.raise_for_status()
feed = feedparser.parse(response.content)

documents = []

for entry in feed.entries:
    title_html = entry.get("title", "")
    body_html = entry.get("summary", "")

    # Use the full post text when the feed provides it.
    content = entry.get("content", [])
    if content:
        body_html = content[0].get("value") or body_html

    # Extract the readable text from the HTML.
    title = BeautifulSoup(title_html, "html.parser").get_text(" ", strip=True)
    body = BeautifulSoup(body_html, "html.parser").get_text(" ", strip=True)

    if body:
        documents.append(f"{title}\n\n{body}".strip())

print(f"Documents collected: {len(documents)}")

if documents:
    print("\nPreview of the first document:")
    print(documents[0][:350])

Documents collected: 25

Preview of the first document:
Reopening the PSF Grants Program with Twice-Yearly Funding Rounds

The Python Software Foundation (PSF) is excited to announce that the PSF Grants Program is reopening with a restructured schedule of two funding rounds per year. After the pause in 2025 , and the special funding round held in 2026 , the PSF is reopening the PSF Grants Program on an 


In [2]:
unique_words = set()

for document in documents:
    words = re.findall(r"\b\w+\b", document.lower())
    unique_words.update(word for word in words if word.isalpha())

document_count = len(documents)
vocabulary_size = len(unique_words)

print(f"Document count: {document_count}")
print(f"Unique word count: {vocabulary_size:,}")

assert document_count >= 20, "The collection needs at least 20 documents."
assert vocabulary_size > 2000, "The collection needs more than 2,000 unique words."

print("Both document collection requirements are met.")

Document count: 25
Unique word count: 2,768
Both document collection requirements are met.


### Our Second Talking Point — Total Words vs. Unique Words

The total word count includes every time a word appears. The unique word count counts each different word once.

For example, “Python software and python tools” contains five words but only four unique words. After we convert the text to lowercase, `Python` and `python` count as the same word. This check helps us see how many different words are in our collection.

In [3]:
example_text = "Python software and python tools"

example_words = re.findall(r"\b\w+\b", example_text.lower())
example_words = [word for word in example_words if word.isalpha()]
example_vocabulary = set(example_words)

print(f"Total words: {len(example_words)}")
print(f"Unique words: {len(example_vocabulary)}")
print(f"Vocabulary: {sorted(example_vocabulary)}")

Total words: 5
Unique words: 4
Vocabulary: ['and', 'python', 'software', 'tools']


## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


### Our Tokenizer

After collecting the documents, we split each document into smaller pieces called tokens. In our example, a token can be a word or a number.

We convert the text to lowercase so that `Python` and `python` are treated as the same token. We also use punctuation and underscores as separators. We keep the tokens in their original order because word order matters when we search for a phrase later.

In [4]:
import re


def tokenize(text):
    # Keep letters and numbers. Treat punctuation and underscores as separators.
    return re.findall(r"[^\W_]+", text.lower())


# Tokenize the first document from Step 1.
tokens = tokenize(documents[0])

print(f"Tokens in the first document: {len(tokens):,}")
print("First 20 tokens:")
print(tokens[:20])

Tokens in the first document: 1,033
First 20 tokens:
['reopening', 'the', 'psf', 'grants', 'program', 'with', 'twice', 'yearly', 'funding', 'rounds', 'the', 'python', 'software', 'foundation', 'psf', 'is', 'excited', 'to', 'announce', 'that']


### Our Second Talking Point — Why Punctuation Matters

People can write `Python!`, `PYTHON`, or `python.`. After tokenization, all three become `python`. This helps the search treat them as the same word.

Our tokenizer also splits `data_science` into `data` and `science` and keeps numbers such as `2026`. The example below shows what happens to each part of the text.

In [5]:
tokenizer_example = "Python! PYTHON, python. Data_science in 2026."
example_tokens = tokenize(tokenizer_example)

print("Original text:", tokenizer_example)
print("Tokens:", example_tokens)

assert example_tokens == [
    "python", "python", "python", "data", "science", "in", "2026"
]

Original text: Python! PYTHON, python. Data_science in 2026.
Tokens: ['python', 'python', 'python', 'data', 'science', 'in', '2026']


## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.



### 🗣 Our Second Talking Point — Why Stop-Word Removal and Stemming Matter

Stop-word removal eliminates common words such as "the", "is", "and", and "in" because they usually provide limited information for search.

Stemming reduces words to their root forms. For example, "running", "runs", and "runner" may be reduced to similar root forms.

We use NLTK's English stop-word list and Porter Stemmer to normalize our tokens. This reduces vocabulary size and helps the search engine recognize related word variations.

**Important:** We preserve the order of the remaining tokens because it is needed later for positional indexing and phrase searching.


In [6]:
import nltk
from functools import lru_cache
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Load the NLTK English stop-word list (download it once if it is missing).
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()
stem = lru_cache(maxsize=None)(stemmer.stem)    # remember results: the same word is stemmed thousands of times


def remove_stop_words(tokens):
    """Stage 1: drop every token that is in the NLTK English stop-word list."""
    return [t for t in tokens if t not in stop_words]


def stem_tokens(tokens):
    """Stage 2: reduce each token to its Porter stem."""
    return [stem(t) for t in tokens]


def normalize_tokens(tokens):
    """The full pipeline, used for documents AND queries: stop words out, then stems."""
    return stem_tokens(remove_stop_words(tokens))


print(f"{len(stop_words)} stop words loaded, e.g.: {sorted(stop_words)[:12]}")

198 stop words loaded, e.g.: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any']


In [7]:

# Demonstrate stop-word removal and suffix stripping

sample_tokens = ["the", "playing", "connected", "running", "and", "happiness"]

print("Original tokens:", sample_tokens)

filtered_tokens = remove_stop_words(sample_tokens)
print("After stop-word removal:", filtered_tokens)

stemmed_tokens = stem_tokens(filtered_tokens)
print("After stemming / suffix stripping:", stemmed_tokens)


Original tokens: ['the', 'playing', 'connected', 'running', 'and', 'happiness']
After stop-word removal: ['playing', 'connected', 'running', 'happiness']
After stemming / suffix stripping: ['play', 'connect', 'run', 'happi']


### Linguistic Processing Pipeline – Observation

In this section, I tested how stop-word removal and stemming work using a few sample words.

I observed that common words like "the" and "and" were removed because they are stop words. The remaining words were reduced to their stems using the Porter Stemmer. For example, "playing" became "play", "connected" became "connect", and "running" became "run".

I also noticed that "happiness" became "happi", which shows that stemming does not always produce a complete dictionary word.

From this activity, I understood that stemming uses rules to remove or change word endings. This helps the search system treat related forms of words more consistently and improves matching between documents and search queries.

In [8]:
import pandas as pd

tokens = tokenize(documents[0])               # BEFORE: tokenizer output
no_stop = remove_stop_words(tokens)           # after stage 1
norm_tokens = normalize_tokens(tokens)        # after stage 2 = final tokens

assert len(norm_tokens) == len(no_stop)       # stemming changes words, never how many there are

print("First document")
print(f"  Tokenizer output:        {len(tokens):>5,} tokens")
print(f"  After stop-word removal: {len(no_stop):>5,} tokens  ({1 - len(no_stop) / len(tokens):.0%} removed)")
print(f"  After stemming:          {len(norm_tokens):>5,} tokens")

print("\nBEFORE (first 20 tokens):", tokens[:20])
print("AFTER  (first 20 tokens):", norm_tokens[:20])

# Token-by-token view of the first 25 tokens
walk = pd.DataFrame({"token": tokens[:25]})
walk["stop word?"] = walk["token"].isin(stop_words)
walk["stem (kept tokens only)"] = [("—" if t in stop_words else stem(t)) for t in walk["token"]]
display(walk)

# The same three stages across the WHOLE collection: how many different terms are left?
all_tokens = [t for text in documents for t in tokenize(text)]
vocab_tokens = set(all_tokens)
vocab_no_stop = set(remove_stop_words(vocab_tokens))
vocab_stems = set(stem_tokens(vocab_no_stop))
sizes = pd.DataFrame({"unique terms": [len(vocab_tokens), len(vocab_no_stop), len(vocab_stems)]},
                     index=["Tokenizer output", "After stop-word removal", "After stemming"])
sizes["vs. tokenizer output"] = (sizes["unique terms"] / len(vocab_tokens) - 1).map("{:+.0%}".format)
display(sizes)

First document
  Tokenizer output:        1,033 tokens
  After stop-word removal:   627 tokens  (39% removed)
  After stemming:            627 tokens

BEFORE (first 20 tokens): ['reopening', 'the', 'psf', 'grants', 'program', 'with', 'twice', 'yearly', 'funding', 'rounds', 'the', 'python', 'software', 'foundation', 'psf', 'is', 'excited', 'to', 'announce', 'that']
AFTER  (first 20 tokens): ['reopen', 'psf', 'grant', 'program', 'twice', 'yearli', 'fund', 'round', 'python', 'softwar', 'foundat', 'psf', 'excit', 'announc', 'psf', 'grant', 'program', 'reopen', 'restructur', 'schedul']


,token,stop word?,stem (kept tokens only)
0,reopening,False,reopen
1,the,True,—
2,psf,False,psf
3,grants,False,grant
4,program,False,program
5,with,True,—
6,twice,False,twice
7,yearly,False,yearli
8,funding,False,fund
9,rounds,False,round


,unique terms,vs. tokenizer output
Tokenizer output,2867,+0%
After stop-word removal,2737,-5%
After stemming,1933,-33%


### Our Second Talking Point — Stage 1: Stop Words Are a Trade-off

Removing stop words makes the index smaller and less noisy, but the NLTK list is generic, so it has two side effects:

- **It removes words that can change the meaning.** The list contains negations such as `not`, `no` and `nor`, so the query `not python` becomes just `python`. A query made only of stop words, such as `to be or not to be`, ends up **empty**.
- **It does not know our collection.** Every collection has its own very common words. If a term appears in almost every document, it cannot help us pick one document over another, even though it is not on the list.

The tokenizer splits `don't` into `don` and `t`, and the NLTK list contains both pieces, so contractions disappear cleanly. We keep the very common terms found in the last part of the cell below instead of removing them: phrase queries such as "python software" need them, and the IDF weight introduced later in the notebook already gives them a low score. If a leftover HTML fragment shows up in that list, it is a data-cleaning issue for Step 1 and not a real word.

In [9]:
from collections import Counter

# 1) Contractions: the tokenizer splits them and NLTK's list contains both pieces.
example = "It's what we don't know"
print(f"{example!r} -> {tokenize(example)} -> {normalize_tokens(tokenize(example))}")

# 2) Words the list removes that can change the meaning.
print("\nNegations in the stop-word list:", sorted({"not", "no", "nor"} & stop_words))
for phrase in ["not python", "to be or not to be"]:
    print(f"{phrase!r} -> {normalize_tokens(tokenize(phrase))}")

# 3) Corpus-specific common terms: the share of documents that contain each term.
doc_terms = [set(normalize_tokens(tokenize(text))) for text in documents]
doc_counts = Counter(term for terms in doc_terms for term in terms)
common = {term: round(n / len(documents), 2) for term, n in doc_counts.most_common() if n / len(documents) >= 0.9}
print(f"\nTerms found in at least 90% of the {len(documents)} documents ({len(common)}):")
print(common)

"It's what we don't know" -> ['it', 's', 'what', 'we', 'don', 't', 'know'] -> ['know']

Negations in the stop-word list: ['no', 'nor', 'not']
'not python' -> ['python']
'to be or not to be' -> []

Terms found in at least 90% of the 25 documents (3):
{'python': 1.0, 'commun': 0.92, 'psf': 0.92}


### Our Second Talking Point — Stage 2: Stemming Merges Words, and Sometimes Too Many

Stemming makes a search more forgiving. `connect`, `connected`, `connecting` and `connection` all become `connect`, so a query for one form also finds documents that use the others. It also shrinks the vocabulary, as the table above showed.

It has three costs:

- **Stems are not always words.** `software` becomes `softwar`, so we cannot show stems to a user as they are, and a query has to be stemmed the same way.
- **Different words can be merged** (over-stemming), which can return documents the user did not mean.
- **Irregular forms are missed.** `ran` and `run` stay apart because the stemmer only cuts endings.

The cell below shows a clean example, then lists the biggest merges in **our** collection so we can look for over-stemming ourselves.

In [10]:
from collections import defaultdict

# 1) A clean example: five surface forms, one stem.
family = ["connect", "connected", "connecting", "connection", "connections"]
print({word: stem(word) for word in family})
assert len({stem(word) for word in family}) == 1

# 2) What does stemming merge in OUR collection? Group the remaining words by their stem.
vocabulary = {t for text in documents for t in remove_stop_words(tokenize(text)) if t.isalpha()}
groups = defaultdict(set)
for word in vocabulary:
    groups[stem(word)].add(word)
merged = {s: sorted(words) for s, words in groups.items() if len(words) > 1}

print(f"\n{len(vocabulary):,} distinct words -> {len(groups):,} stems; {len(merged):,} stems merge two or more words.")
print("Biggest merges (check them for words that should not be together):")
for s, words in sorted(merged.items(), key=lambda item: -len(item[1]))[:8]:
    print(f"  {s!r:<12} <- {words}")

# 3) What stemming cannot do: irregular forms stay apart.
print("\nIrregular forms:", {word: stem(word) for word in ["ran", "run", "better", "good"]})

{'connect': 'connect', 'connected': 'connect', 'connecting': 'connect', 'connection': 'connect', 'connections': 'connect'}

2,639 distinct words -> 1,835 stems; 517 stems merge two or more words.
Biggest merges (check them for words that should not be together):
  'organ'      <- ['organization', 'organizations', 'organize', 'organized', 'organizer', 'organizers', 'organizes', 'organizing']
  'support'    <- ['support', 'supported', 'supporter', 'supporters', 'supporting', 'supportive', 'supports']
  'improv'     <- ['improve', 'improved', 'improvement', 'improvements', 'improves', 'improving']
  'maintain'   <- ['maintain', 'maintained', 'maintainer', 'maintainers', 'maintaining', 'maintains']
  'commun'     <- ['communicate', 'communicated', 'communication', 'communications', 'communities', 'community']
  'continu'    <- ['continually', 'continue', 'continued', 'continues', 'continuing', 'continuous']
  'develop'    <- ['develop', 'developed', 'developer', 'developers', 'developing',

## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.




### 🔧 Your Task:

- Build the inverted index using a dictionary (`term → document IDs`).
- Build a positional index (`term → document ID → positions`).
- Implement phrase searching using positional indexing.
- Test two multi-word phrase queries and display the matching document IDs.


In [11]:
from collections import defaultdict

def build_inverted_index(documents):
    index = defaultdict(list)
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        seen = set()
        for token in tokens:
            if token not in seen:
                index[token].append(doc_id)
                seen.add(token)
    return dict(index)

inverted_index = build_inverted_index(documents)
print(dict(list(inverted_index.items())[:10]))  # Preview first 10 terms


{'reopen': [0], 'psf': [0, 1, 2, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24], 'grant': [0, 1, 2, 4, 5, 7, 8, 13, 16, 19, 21, 22, 23], 'program': [0, 1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 16, 18, 19, 20, 21, 22, 23], 'twice': [0], 'yearli': [0, 10], 'fund': [0, 1, 2, 3, 7, 13, 18, 19, 20, 21, 22, 23, 24], 'round': [0, 2, 7, 13], 'python': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24], 'softwar': [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14, 16, 19, 22, 23]}


In [12]:

from collections import defaultdict

def build_positional_index(documents):

    positional_index = defaultdict(lambda: defaultdict(list))

    for doc_id, document in enumerate(documents):

        # Apply Person 2's tokenization and normalization
        tokens = tokenize(document)
        normalized_tokens = normalize_tokens(tokens)

        # Store each term's position in the document
        for position, term in enumerate(normalized_tokens):
            positional_index[term][doc_id].append(position)

    return positional_index


# Build the positional index
positional_index = build_positional_index(documents)

print("Positional index created successfully!")
print("Total unique terms:", len(positional_index))


Positional index created successfully!
Total unique terms: 1933


In [13]:
# Inspect the positions of a sample term

sample_term = "python"

print(f"Term: {sample_term}")
print("-" * 50)

if sample_term in positional_index:
    for doc_id, positions in positional_index[sample_term].items():
        print(f"Document ID: {doc_id}")
        print(f"Positions: {positions}")
        print()
else:
    print("Term not found.")

Term: python
--------------------------------------------------
Document ID: 0
Positions: [8, 50, 195, 413, 449, 543, 575, 578, 589, 598, 602, 615, 625]

Document ID: 1
Positions: [84, 92, 466, 477, 488, 497, 501, 514]

Document ID: 2
Positions: [0, 9, 16, 21, 43, 56, 67, 87, 93, 101, 112, 182, 190, 203, 210, 230, 237, 258, 266, 275, 279, 328, 343, 345, 364, 373, 389, 401, 507, 564, 578, 627, 641, 708, 767, 842]

Document ID: 3
Positions: [0, 32, 45, 48, 54, 59, 105, 124, 147, 153, 158, 191, 202, 209, 216, 241, 259, 272, 311, 325, 341, 357, 369, 391, 404, 482]

Document ID: 4
Positions: [6, 80, 145, 147, 150, 161, 170, 174, 187]

Document ID: 5
Positions: [10, 38, 104, 115, 124, 135, 161, 238, 241, 252, 261, 265, 278]

Document ID: 6
Positions: [29, 58, 65, 77, 104, 116, 128, 158, 168, 190, 209, 232, 246, 254, 291, 302, 312, 335, 405, 440, 450, 470, 501, 536, 554, 567, 578, 589, 614, 826, 830, 843]

Document ID: 7
Positions: [15, 25, 121, 126, 143, 163, 306, 328]

Document ID: 8
Positi

In [14]:

# Boolean Retrieval: AND and AND NOT

def boolean_and(term1, term2, index):
    docs1 = set(index.get(term1, []))
    docs2 = set(index.get(term2, []))
    return sorted(docs1 & docs2)

def boolean_and_not(term1, term2, index):
    docs1 = set(index.get(term1, []))
    docs2 = set(index.get(term2, []))
    return sorted(docs1 - docs2)

# Test both queries
print("python AND softwar:",
      boolean_and("python", "softwar", inverted_index))

print("python AND NOT softwar:",
      boolean_and_not("python", "softwar", inverted_index))


python AND softwar: [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14, 16, 19, 22, 23]
python AND NOT softwar: [7, 10, 12, 15, 17, 18, 20, 21, 24]


### Boolean Retrieval – AND and AND NOT

In this section, I tested Boolean retrieval using the terms "python" and "softwar".

For the AND query, I received 16 documents because both terms were present in those documents.

For the AND NOT query, I received 9 documents because they contained "python" but did not contain "softwar".

I understood that AND helps us narrow down the search results by finding documents containing both terms, while AND NOT helps exclude documents containing an unwanted term.

These operations make searching more specific and help retrieve documents based on the user's requirements.

## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries.
- Demonstrate that they return the correct documents.


In [15]:
# Placeholder for phrase query implementation
# You may build a position-aware index or use string search within docs after normalization

# Example:
query1 = "Python software"
query2 = "Python community"

def phrase_query(query, documents):
    results = []
    query_tokens = normalize_tokens(tokenize(query))
    for doc_id, doc in enumerate(documents):
        doc_tokens = normalize_tokens(tokenize(doc))
        # Search for the sequence in the token list
        for i in range(len(doc_tokens) - len(query_tokens) + 1):
            if doc_tokens[i:i+len(query_tokens)] == query_tokens:
                results.append(doc_id)
                break
    return results

# Run phrase queries on the loaded documents
results1 = phrase_query(query1, documents)
results2 = phrase_query(query2, documents)

print(f"Documents containing the phrase '{query1}': {results1}")
print(f"Documents containing the phrase '{query2}': {results2}")

Documents containing the phrase 'Python software': [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14, 22]
Documents containing the phrase 'Python community': [0, 1, 2, 4, 5, 6, 7, 9, 11, 13, 14, 16, 17, 19, 20, 21, 22, 23, 24]


In [16]:
# Find common consecutive word pairs in the documents

from collections import Counter

biword_counts = Counter()

for document in documents:
    words = normalize_tokens(tokenize(document))
    for i in range(len(words) - 1):
        pair = (words[i], words[i + 1])
        biword_counts[pair] += 1

print("Most common consecutive word pairs:")
for pair, count in biword_counts.most_common(10):
    print(pair, ":", count)

Most common consecutive word pairs:
('python', 'commun') : 74
('psf', 'board') : 70
('python', 'org') : 29
('open', 'sourc') : 27
('board', 'director') : 27
('pycon', 'us') : 26
('softwar', 'foundat') : 24
('python', 'softwar') : 23
('grant', 'program') : 19
('python', 'packag') : 19


## 🧠 Additional Challenge: Implement Positional Indexes

Implement Positional Indexes, an advanced version of an inverted index that not only stores which documents a term appears in, but also where (at what positions) the term occurs within each document.

Apply it to the documents you used to produce the Inverted Matrix during the Active Learning Workshop. 

Then compare it against a **Term Document Count Matrix** (which you don't have to implement)

And finally, fill in the blanks in the table below.

| Term | What is it? | How is it used? | Pros | Cons | 
|----------------|-------------|-------------|-------------|-------------|
| Term Frequency (TF)  |  |  |  |
| Inverse Document Frequency (idF weight)  |  |  |  |

Then use the table to prepare talking points on:
- Which implementation you prefer to use for searching bigrams (a.k.a., biwords), pairs of consecutive words in a document, and why?

#### Sample solution:

In [17]:
# Implement Positional Indexes: Store term positions for each document
from collections import defaultdict

def build_positional_index(documents):
    positional_index = defaultdict(lambda: defaultdict(list))
    for doc_id, text in enumerate(documents):
        tokens = normalize_tokens(tokenize(text))
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)
    return positional_index

# Build positional index for the loaded documents
positional_index = build_positional_index(documents)

# Example: Show positions for a sample term
sample_term = 'python'  # Change to any term you want to inspect
print(f"Positions for term '{sample_term}':")
for doc_id, positions in positional_index[sample_term].items():
    print(f"  Document {doc_id}: {positions}")

Positions for term 'python':
  Document 0: [8, 50, 195, 413, 449, 543, 575, 578, 589, 598, 602, 615, 625]
  Document 1: [84, 92, 466, 477, 488, 497, 501, 514]
  Document 2: [0, 9, 16, 21, 43, 56, 67, 87, 93, 101, 112, 182, 190, 203, 210, 230, 237, 258, 266, 275, 279, 328, 343, 345, 364, 373, 389, 401, 507, 564, 578, 627, 641, 708, 767, 842]
  Document 3: [0, 32, 45, 48, 54, 59, 105, 124, 147, 153, 158, 191, 202, 209, 216, 241, 259, 272, 311, 325, 341, 357, 369, 391, 404, 482]
  Document 4: [6, 80, 145, 147, 150, 161, 170, 174, 187]
  Document 5: [10, 38, 104, 115, 124, 135, 161, 238, 241, 252, 261, 265, 278]
  Document 6: [29, 58, 65, 77, 104, 116, 128, 158, 168, 190, 209, 232, 246, 254, 291, 302, 312, 335, 405, 440, 450, 470, 501, 536, 554, 567, 578, 589, 614, 826, 830, 843]
  Document 7: [15, 25, 121, 126, 143, 163, 306, 328]
  Document 8: [4, 10, 64, 131, 133, 223, 245, 285, 348]
  Document 9: [1, 28, 54, 60, 65, 68, 74, 78, 89, 123, 129, 151]
  Document 10: [0, 11, 17, 30, 45, 60, 

In [18]:

# Build a Biword Index using the existing tokenizer and normalization

from collections import defaultdict

biword_index = defaultdict(set)

for doc_id, document in enumerate(documents):
    tokens = normalize_tokens(tokenize(document))

    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        biword_index[pair].add(doc_id)

# Search two-word phrases using the biword index
def search_biword(phrase):
    tokens = normalize_tokens(tokenize(phrase))

    if len(tokens) != 2:
        return []

    return sorted(biword_index.get(tuple(tokens), set()))

print("Python software:", search_biword("Python software"))
print("Python community:", search_biword("Python community"))


Python software: [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14, 22]
Python community: [0, 1, 2, 4, 5, 6, 7, 9, 11, 13, 14, 16, 17, 19, 20, 21, 22, 23, 24]


### Biword Index – Observation

In this section, I created a biword index to store consecutive word pairs along with their document IDs.

When I searched for "Python software", I received 13 matching documents. For "Python community", I received 19 matching documents.

I observed that the biword index helps us find documents containing specific two-word combinations without searching through every document again.

I also understood that a positional index stores the position of each individual word, while a biword index stores pairs of consecutive words. Both can support phrase searches, but the positional index is more flexible for searching longer phrases.

These results are based on normalized tokens, so the matched words may not have been directly next to each other in the original text.

In [19]:

# Search for phrases using the existing positional index

def positional_phrase_search(phrase, positional_index):
    terms = normalize_tokens(tokenize(phrase))

    if not terms:
        return []

    # Find documents containing every term
    common_docs = set(positional_index.get(terms[0], {}).keys())

    for term in terms[1:]:
        common_docs &= set(positional_index.get(term, {}).keys())

    matching_docs = []

    # Check whether terms occur in consecutive positions
    for doc_id in common_docs:
        first_positions = positional_index[terms[0]][doc_id]

        for start_pos in first_positions:
            if all(
                start_pos + offset in positional_index[term][doc_id]
                for offset, term in enumerate(terms[1:], start=1)
            ):
                matching_docs.append(doc_id)
                break

    return sorted(matching_docs)


# Test two phrases
print("Python software:",
      positional_phrase_search("Python software", positional_index))

print("Python community:",
      positional_phrase_search("Python community", positional_index))


Python software: [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 13, 14, 22]
Python community: [0, 1, 2, 4, 5, 6, 7, 9, 11, 13, 14, 16, 17, 19, 20, 21, 22, 23, 24]


### Positional Phrase Search – Observation

In this section, I used the positional index to search for two phrases, "Python software" and "Python community".

The first phrase returned 13 documents, while the second returned 19 documents. These results matched the biword index results.

I understood that the positional index checks whether the words appear in the correct order and at consecutive positions. For example, if "python" is at position 5, "softwar" should be at position 6 for the phrase to match.

I observed that the biword index is useful for searching two-word combinations, while the positional index is more flexible because it can also search longer phrases.

Both methods are helpful for retrieving relevant documents, depending on the type of query.

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

###  Observation – Matrix Comparison

From this comparison, I understood that a positional index is more useful when we want to search for two words appearing together. For example, if we search for "Python software", the positional index can check whether "software" appears immediately after "Python" in a document.

A Term Document Count Matrix only tells us how many times each word appears. It cannot tell us whether the two words appear next to each other.

I would prefer the positional index for bigram searches because it stores the position of each word and helps us find matching phrases without checking every document manually.

### 📝 Completed Table: Term Frequency (TF) and Inverse Document Frequency (IDF weight)

| Term                           | What is it?                                                                 | How is it used?                          | Pros                                   | Cons                                   |
|-------------------------------|----------------------------------------------------------------------------|------------------------------------------|----------------------------------------|----------------------------------------|
| Term Frequency (TF)            | Number of times a term appears in a document                               | Measures term importance in a document   | Simple, fast, good for ranking         | Doesn't account for common terms       |
| Inverse Document Frequency     | Log ratio of total docs to docs containing the term                        | Weights rare terms higher                | Reduces impact of common words         | Can overweight rare, irrelevant terms  |
| (IDF weight)                   | Used with TF to score term relevance: TF-IDF = TF * IDF                    | Improves search relevance                | Highlights unique, important terms     | Requires global corpus statistics      |

Let's implement the Inverse Document Frequency (IDF)

In [20]:
# Calculate Inverse Document Frequency (IDF) for each term using the positional index
import math

def compute_idf_from_positional_index(positional_index, total_docs):
    idf_scores = {}
    for term, doc_positions in positional_index.items():
        doc_count = len(doc_positions)
        if doc_count > 0:
            # --- IGNORE ---
            # print(f"Computing IDF for {term} '{total_docs}' / {doc_count}")
            idf_scores[term] = math.log(total_docs / doc_count)
        else:
            idf_scores[term] = 0.0
    return idf_scores

# Compute IDF for all terms using the positional index
idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

# Example: Show IDF for a sample term
# NOTE : We applied stemming in the normalization step, so the term may not match exactly if it was stemmed
sample_term = 'softwar'  # Change to any term you want to inspect
print(f"IDF for term '{sample_term}': {idf_scores.get(sample_term, 0.0)}")

IDF for term 'softwar': 0.44628710262841953


### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

In [21]:
# Calculate Term Frequency (TF) for the term 'softwar' in each document
# TF is simply the number of times the term appears in a document
import pandas as pd

term = 'softwar'  # Use the stemmed term as before

tf_values = []
for doc_id, doc in enumerate(documents):
    tokens = normalize_tokens(tokenize(doc))
    tf = tokens.count(term)
    tf_values.append(tf)
    print(f"TF for term '{term}' in Document {doc_id}: {tf}")

# Display as a DataFrame for clarity
TF_df = pd.DataFrame({'Document': list(range(len(documents))), 'TF': tf_values})
print(TF_df)

TF for term 'softwar' in Document 0: 3
TF for term 'softwar' in Document 1: 2
TF for term 'softwar' in Document 2: 1
TF for term 'softwar' in Document 3: 6
TF for term 'softwar' in Document 4: 3
TF for term 'softwar' in Document 5: 6
TF for term 'softwar' in Document 6: 3
TF for term 'softwar' in Document 7: 0
TF for term 'softwar' in Document 8: 1
TF for term 'softwar' in Document 9: 1
TF for term 'softwar' in Document 10: 0
TF for term 'softwar' in Document 11: 1
TF for term 'softwar' in Document 12: 0
TF for term 'softwar' in Document 13: 1
TF for term 'softwar' in Document 14: 1
TF for term 'softwar' in Document 15: 0
TF for term 'softwar' in Document 16: 1
TF for term 'softwar' in Document 17: 0
TF for term 'softwar' in Document 18: 0
TF for term 'softwar' in Document 19: 2
TF for term 'softwar' in Document 20: 0
TF for term 'softwar' in Document 21: 0
TF for term 'softwar' in Document 22: 1
TF for term 'softwar' in Document 23: 1
TF for term 'softwar' in Document 24: 0
    Docume

#### 📊 Example: Term Frequency (TF) and TF-IDF Calculation for 'softwar' in Document 0

TF for term 'softwar' in Document 0:

$$
\text{TF}(\text{softwar}, 0) = n_{\text{softwar}, 0}
$$

TF-IDF for term 'softwar' in Document 0:

$$
\text{TF-IDF}(\text{softwar}, 0) = \text{TF}(\text{softwar}, 0) \times \text{IDF}(\text{softwar}) = n_{\text{softwar}, 0} \times \text{IDF}(\text{softwar})
$$

Where:
- $n_{\text{softwar}, 0}$ is the number of times 'softwar' appears in Document 0.
- $\text{IDF}(\text{softwar})$ is the previously calculated IDF value for 'softwar'.

Using the actual values from our output:

- TF for 'softwar' in Document 0 = 3
- IDF for 'softwar' = 0.446287

Therefore,

$$
\text{TF-IDF} = 3 \times 0.446287 = 1.338861
$$

This means the TF-IDF score for 'softwar' in Document 0 is approximately 1.3389. The score helps us understand the importance of this term in the document and compare it with other documents.

In [22]:

# Calculate TF-IDF for 'softwar' in all documents

term = "softwar"

idf = idf_scores.get(term, 0)

tfidf_values = []

for tf in tf_values:
    tfidf = tf * idf
    tfidf_values.append(tfidf)

tfidf_df = pd.DataFrame({
    "Document": range(len(tf_values)),
    "TF": tf_values,
    "IDF": [idf] * len(tf_values),
    "TF-IDF": tfidf_values
})

tfidf_df = tfidf_df.sort_values(
    by="TF-IDF",
    ascending=False
)

print("TF-IDF Ranking for:", term)
display(tfidf_df.head(10))


TF-IDF Ranking for: softwar


,Document,TF,IDF,TF-IDF
3,3,6,0.446287,2.677723
5,5,6,0.446287,2.677723
0,0,3,0.446287,1.338861
6,6,3,0.446287,1.338861
4,4,3,0.446287,1.338861
1,1,2,0.446287,0.892574
19,19,2,0.446287,0.892574
8,8,1,0.446287,0.446287
2,2,1,0.446287,0.446287
16,16,1,0.446287,0.446287


## 🧠 Reflection

###  Reflection – TF-IDF and Document Ranking

While working on the TF-IDF calculation, I learned how term frequency and inverse document frequency help rank documents. I observed that Documents 3 and 5 received the highest score because the term "softwar" appeared six times in each document.

I also understood the difference between a positional index and a term-document count matrix. A positional index is useful for checking whether words appear together, while a term-document count matrix helps us compare word frequencies.

Overall, this workshop helped me understand how search systems organize information and find relevant documents.

In [23]:
# Combine TF and IDF to compute TF-IDF for the term 'softwar' in each document
# this is where the user left off
def query_processor(query, inverted_index):
    results = []
    for term in query.split():
        if term in inverted_index:
            results.extend(inverted_index[term])
    return list(set(results))  # ✅ return a list



### TF-IDF Calculation and Document Ranking – Observation

From the results, I observed that Documents 3 and 5 have the highest TF-IDF score of 2.6777 because the term "softwar" appears 6 times in each document.

Documents 0, 4, and 6 have a TF-IDF score of 1.3389 because the term appears 3 times in each document.

The IDF value is the same for all documents (0.4463) because we are calculating the importance of the same word across the collection.

I understood that TF-IDF helps us identify and rank documents based on how important a search term is within them. In this example, documents with a higher frequency of "softwar" receive higher scores and appear at the top of the ranking.

### Applications of Information Retrieval in RAG and AI Agents

**Retrieval-Augmented Generation (RAG)**

I understood that RAG combines information retrieval with AI-generated answers. Instead of answering only from its existing knowledge, an AI model can first retrieve relevant information from documents and use it to prepare a response.

For example, if a user asks about Python software, our inverted index can identify documents containing the search terms, and TF-IDF can help rank those documents. The most relevant document content can then be provided to an AI model to generate an answer.

**AI Agents**

AI agents can use information retrieval to find information needed to complete a task. For example, an AI agent could search our Python blog collection for articles about Python community activities and summarize the relevant results.

The inverted index helps identify matching documents, while TF-IDF can help rank them.

**My Observation**

From this workshop, I understood that information retrieval is useful beyond traditional search engines. Techniques like inverted indexes, positional indexes, and TF-IDF can also support RAG systems and AI agents by helping them find relevant information before generating answers or completing tasks.

## 🧠 Push your code to the course GitHub repository

Update your team number and then run the code. 

In [24]:
!python submit_assignment.py --notebook IR_InvertedMatrix_Workshop.ipynb --student_id "teamA"


python: can't open file 'c:\\Conestoga_Projects\\Machine Learning Programing\\IR-invertedmatrix-workshop\\submit_assignment.py': [Errno 2] No such file or directory
